<a href="https://colab.research.google.com/github/jihun0251/korean-spam-classifierML/blob/main/URL_%ED%8A%B9%EC%A7%95_%EC%B6%94%EC%B6%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. URL 문자열 자체에서 뽑는 특징 (어휘적 feature)

## URL 쪼개기
- 스키마, 도메인, 포트, 경로, 쿼리 등으로 URL을 쪼갬

In [1]:
from urllib.parse import urlparse

url = "https://login.kb-bank.xyz:8080/verify?id=123"
parsed = urlparse(url)

print("scheme:", parsed.scheme)     # https
print("netloc:", parsed.netloc)     # login.kb-bank.xyz:8080
print("hostname:", parsed.hostname) # login.kb-bank.xyz
print("path:", parsed.path)         # /verify
print("query:", parsed.query)       # id=123
print("port:", parsed.port)         # 8080

scheme: https
netloc: login.kb-bank.xyz:8080
hostname: login.kb-bank.xyz
path: /verify
query: id=123
port: 8080


## 각 요소들에서 특징 추출

In [2]:
from urllib.parse import urlparse
import re

def extract_url_features(url):
    parsed = urlparse(url)
    hostname = parsed.hostname or ""

    suspicious_words = ["login", "verify", "secure", "account", "update", "bank"]
    shorteners = ["bit.ly", "tinyurl", "goo.gl", "t.co", "is.gd"]

    features = {
        "url_length": len(url),
        "dot_count": hostname.count("."),
        "is_https": 1 if parsed.scheme == "https" else 0,
        "is_ip": 1 if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", hostname) else 0,
        "has_at": 1 if "@" in url else 0,
        "keyword_count": sum(1 for w in suspicious_words if w in url.lower()),
        "is_shortened": 1 if any(s in url.lower() for s in shorteners) else 0,
        "special_count": sum(url.count(c) for c in ["-", "%", "=", "&", "_"]),
    }
    return features

# 테스트: 위험해 보이는 URL vs 정상 URL
print(extract_url_features("http://login.kb-bank.xyz/verify?id=123"))
print(extract_url_features("https://www.naver.com"))

{'url_length': 38, 'dot_count': 2, 'is_https': 0, 'is_ip': 0, 'has_at': 0, 'keyword_count': 3, 'is_shortened': 0, 'special_count': 2}
{'url_length': 21, 'dot_count': 2, 'is_https': 1, 'is_ip': 0, 'has_at': 0, 'keyword_count': 0, 'is_shortened': 0, 'special_count': 0}


# 2. 도메인 평판·이력에서 뽑는 특징 (외부 조회 feature)

## 도메인 나이 — python-whois 라이브러리

In [3]:
!pip install python-whois
import whois
from datetime import datetime

def get_domain_age_days(domain):
    try:
        info = whois.whois(domain)
        created = info.creation_date
        # 가끔 리스트로 반환되므로 첫 번째 값 사용
        if isinstance(created, list):
            created = created[0]
        if created is None:
            return -1  # 조회 실패
        age = (datetime.now() - created).days
        return age
    except Exception:
        return -1  # 조회 실패 시 -1

print(get_domain_age_days("naver.com"))   # 큰 숫자 (오래된 도메인)
print(get_domain_age_days("google.com"))  # 큰 숫자

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 5.6 MB/s eta 0:00:00


ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


-1
-1


## Google Safe Browsing - 무료 API

In [4]:
import requests

def check_safe_browsing(url, api_key):
    endpoint = "https://safebrowsing.googleapis.com/v4/threatMatches:find"
    payload = {
        "client": {"clientId": "qr-checker", "clientVersion": "1.0"},
        "threatInfo": {
            "threatTypes": ["MALWARE", "SOCIAL_ENGINEERING"],
            "platformTypes": ["ANY_PLATFORM"],
            "threatEntryTypes": ["URL"],
            "threatEntries": [{"url": url}],
        },
    }
    try:
        resp = requests.post(f"{endpoint}?key={api_key}", json=payload, timeout=5)
        result = resp.json()
        # 위협이 발견되면 'matches' 키가 있음
        return 1 if "matches" in result else 0
    except Exception:
        return -1

# 사용: check_safe_browsing("http://의심url.com", "발급받은_API_키")

## SSL 인증서 — 파이썬 내장 ssl

In [5]:
import ssl, socket
from datetime import datetime

def check_ssl(domain):
    try:
        ctx = ssl.create_default_context()
        with ctx.wrap_socket(socket.socket(), server_hostname=domain) as s:
            s.settimeout(5)
            s.connect((domain, 443))
            cert = s.getpeercert()
        # 인증서를 성공적으로 받았으면 유효
        return 1
    except Exception:
        return 0  # 인증서 없음/오류 = 의심

print(check_ssl("naver.com"))  # 1 (유효)

1


# 1번 특징 + 2번 특징 통합 코드

In [18]:

import requests
from datetime import datetime, timezone

def get_domain_age_days_rdap(domain):
    try:
        # rdap.org가 적절한 RDAP 서버로 연결해주는 통합 진입점
        url = f"https://rdap.org/domain/{domain}"
        resp = requests.get(url, timeout=30)

        if resp.status_code != 200:
            return -1  # 조회 실패 (도메인 없음 등)

        data = resp.json()

        # events 목록에서 'registration'(등록) 시점을 찾는다
        created = None
        for event in data.get("events", []):
            if event.get("eventAction") == "registration":
                created = event.get("eventDate")
                break

        if created is None:
            return -1

        # ISO 형식 문자열 → datetime 변환 (끝의 Z는 UTC 표시)
        created_dt = datetime.fromisoformat(created.replace("Z", "+00:00"))
        age = (datetime.now(timezone.utc) - created_dt).days
        return age

    except Exception as e:
        print(f"  (RDAP 오류: {e})")
        return -1

# 테스트
for domain in ["naver.com", "google.com", "github.com"]:
    age = get_domain_age_days_rdap(domain)
    print(f"{domain}: {age}일")

naver.com: 10514일
google.com: 10511일
github.com: 6834일


In [19]:
def extract_all_features(url):
    parsed = urlparse(url)
    domain = parsed.hostname or ""

    features = extract_url_features(url)        # 1번: 문자열 특징
    features["domain_age"] = get_domain_age_days_rdap(domain)  # 2번: 도메인 나이
    features["has_ssl"] = check_ssl(domain)              # 2번: SSL
    # features["safe_browsing"] = ...  # API 키 준비되면 추가

    return features

# 테스트(실제 URL 넣어보기)

In [22]:
test_urls = [
    "https://www.naver.com",
    "http://login-kbbank.xyz/verify",  # 의심스러운 가짜
]

for url in test_urls:
    print(f"\nURL: {url}")
    for key, value in extract_all_features(url).items():
        print(f"  {key}: {value}")

        # 왜 네이버 도메인 나이가 -1로 뜨지..?


URL: https://www.naver.com
  url_length: 21
  dot_count: 2
  is_https: 1
  is_ip: 0
  has_at: 0
  keyword_count: 0
  is_shortened: 0
  special_count: 0
  domain_age: -1
  has_ssl: 1

URL: http://login-kbbank.xyz/verify
  url_length: 30
  dot_count: 1
  is_https: 0
  is_ip: 0
  has_at: 0
  keyword_count: 3
  is_shortened: 0
  special_count: 1
  domain_age: -1
  has_ssl: 0
